### Infer signaling paths beween ligand(s) and target(s) of interest

To determine signaling paths between a ligand and target of interest, we
look at which transcription factors are best regulating the target genes
and are most closely downstream of the ligand (based on the weights of
the edges in the integrated ligand-signaling and gene regulatory
networks). Then, the shortest paths between these transcription factors
and the ligand of interests are determined and genes forming part in
this path are considered as important signaling mediators. Finally, we
look in our collected data source networks for all interactions between
the ligand, signaling mediators, transcription factors and target genes.
This allows to both prioritize signaling mediators and check which of
all collected data sources support the ligand-target predictions of
interest.

For this analysis, you need to define:

- one or more ligands of interest
- one or more target genes of interest

In this vignette, we will demonstrate how to infer signaling paths
between a CAF-ligand (CAF = cancer-associated fibroblast) of interest
and some of its top-predicted p-EMT target genes. The output of this
analysis can be easily imported into Cytoscape for exploration of the
networks.

In [1]:
from nichenetpy.utils import read_csv_cols
from nichenetpy.visualization import get_ligand_signaling_path

import os
import requests
import pickle
import pandas as pd

In [2]:
model_path = os.path.normpath("./tutorial_files")
if not os.path.exists(model_path):
    os.makedirs(model_path)
for filename in (
    ["ltf_matrix.pkl", "nichenet_human.pkl"]
):
    file_path = os.path.join(model_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14944315/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv"
):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
with open("./tutorial_files/ltf_matrix.pkl", "rb") as file:
    ltf_matrix = pickle.loads(file.read())
row_names = ltf_matrix["row_names"]
col_names = ltf_matrix["col_names"]
ltf_matrix = ltf_matrix["mat"]
with open("./tutorial_files/nichenet_human.pkl", "rb") as file:
    model = pickle.loads(file.read())
lr_sig = pd.DataFrame(model["lr_sig"]._mapping, columns=["from", "to", "weight"])
gr = pd.DataFrame(model["gr"]._mapping, columns=["from", "to", "weight"])
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

As an example, we will infer signaling paths between the CAF-ligand TGFB2 and its top-predicted p-EMT target genes SERPINE1 and COL1A1. For better visualization of edge weights, we will also normalize edge weights to make them comparable between signaling and gene regulatory interactions

In [4]:
ligands_oi = ["TGFB2"]
targets_oi = ["SERPINE1", "COL1A1"]
tf_signaling, tf_regulatory = get_ligand_signaling_path(
    ltf_matrix,
    ligands_oi,
    targets_oi,
    row_names,
    col_names,
    lr_sig,
    gr,
    top_n_regulators=4,
    minmax_scaling=True
)

c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\scipy\sparse\_index.py:108: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


KeyboardInterrupt: 